In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/e_commerce_database/'

# 2. Load Raw Datasets
customers = pd.read_csv(os.path.join(base_path, 'customers.csv'))
orders = pd.read_csv(os.path.join(base_path, 'orders.csv'))
products = pd.read_csv(os.path.join(base_path, 'products.csv'))
payments = pd.read_csv(os.path.join(base_path, 'payments.csv'))

df_merged = orders.merge(customers, on='CustomerID', how='left') \
                  .merge(payments, on='OrderID', how='left') \
                  .merge(products, on='ProductID', how='left')

# ==========================================
# FILE 1: RFM Customer Segments & Churn
# ==========================================
df_merged['OrderDate'] = pd.to_datetime(df_merged['OrderDate'])
snapshot_date = df_merged['OrderDate'].max() + pd.Timedelta(days=1)

rfm = df_merged.groupby('CustomerID').agg({
    'OrderDate': lambda x: (snapshot_date - x.max()).days,
    'OrderID': 'nunique',
    'Quantity': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

churn_thresh = rfm['Recency'].quantile(0.75)
rfm['Is_Churned'] = (rfm['Recency'] > churn_thresh).astype(int)
rfm.to_csv(os.path.join(base_path, '01_rfm_customer_segments_and_churn.csv'), index=False)

# ==========================================
# FILE 2: Market Basket Matrix Summary
# ==========================================
basket_summary = orders.groupby(['OrderID', 'ProductID'])['Quantity'].sum().unstack(fill_value=0)
basket_summary.to_csv(os.path.join(base_path, '02_market_basket_matrix.csv'))

# ==========================================
# FILE 3: 30-Day Revenue Forecast
# ==========================================
daily_sales = orders.groupby(pd.to_datetime(orders['OrderDate']).dt.date)['Quantity'].sum().reset_index()
daily_sales['OrderDate'] = pd.to_datetime(daily_sales['OrderDate'])
daily_sales = daily_sales.set_index('OrderDate').asfreq('D')

ts_model = ExponentialSmoothing(daily_sales['Quantity'], trend='add', seasonal='add', seasonal_periods=7).fit()
future_forecast = ts_model.forecast(steps=30)
forecast_df = pd.DataFrame({'Forecast_Date': future_forecast.index, 'Projected_Quantity': future_forecast.values})
forecast_df.to_csv(os.path.join(base_path, '03_30_day_revenue_forecast.csv'), index=False)

# ==========================================
# FILE 4: Cohort Retention Matrix
# ==========================================
orders['OrderMonth'] = pd.to_datetime(orders['OrderDate']).apply(lambda x: pd.Timestamp(x.year, x.month, 1))
orders['CohortMonth'] = orders.groupby('CustomerID')['OrderMonth'].transform('min')

orders['CohortIndex'] = (orders['OrderMonth'].dt.year - orders['CohortMonth'].dt.year) * 12 + \
                       (orders['OrderMonth'].dt.month - orders['CohortMonth'].dt.month) + 1

cohort_counts = orders.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index().pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
retention = cohort_counts.divide(cohort_counts.iloc[:, 0], axis=0)
retention.to_csv(os.path.join(base_path, '04_customer_retention_cohorts.csv'))

# ==========================================
# FILE 5: Payment Friction Summary
# ==========================================
payment_summary = orders.groupby(['PaymentMethod', 'Status']).size().unstack(fill_value=0)
payment_summary.to_csv(os.path.join(base_path, '05_payment_friction_summary.csv'))

print("All 5 processed CSV files successfully generated and uploaded to Google Drive!")

Mounted at /content/drive
All 5 processed CSV files successfully generated and uploaded to Google Drive!
